In [1]:
# Install dependencies (runs once per Colab session)
!pip install -q openai langchain-groq python-dotenv ipywidgets

In [5]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from langchain_groq import ChatGroq
# Load environment variables
load_dotenv(find_dotenv())
# Load API key from Google Colab Secrets
if not os.environ.get("GROQ_API_KEY"):
    try:
      from google.colab import userdata
      os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
      print("Successfully retrieved GROQ_API_KEY from Google Colab Secrets!")
    except Exception:
      pass
api_key = os.environ.get("GROQ_API_KEY","").strip()

if not api_key:
    raise ValueError("GROQ_API_KEY is missing! Please configure it in Colab Secrets.")

# Model selection
MODEL_NAME = "openai/gpt-oss-120b"
client = OpenAI(
      api_key=api_key,
      base_url="https://api.groq.com/openai/v1"
)
# Initialize LangChain ChatGroq
llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    groq_api_key=api_key
)
print(f"Groq client connected successfully! Target Model: {MODEL_NAME}")

Groq client connected successfully! Target Model: openai/gpt-oss-120b


In [11]:
messages = [
      { "role": "system","content": "Role: Shop assistant. Facts: We sell stationery. Rules: Be polite. Format: Reply briefly."},
      {"role": "user","content": "Do you sell notebooks?"}
]

response = client.chat.completions.create(
  model=MODEL_NAME,
  messages=messages,
  temperature=0.2
)
new_message = "The cake was damaged"

messages = [
    {"role": "system", "content": "Classify messages as order, question or complaint."},
    {"role": "user", "content": "I need 2 plum cakes"},
    {"role": "assistant", "content": "order"},
    {"role": "user", "content": "Do you have eggless cakes?"},
    {"role": "assistant", "content": "question"},
    {"role": "user", "content": "My cake was damaged"},
    {"role": "assistant", "content": "complaint"},
    {"role": "user", "content": new_message}
]
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=messages,
    temperature=0.2
)
print(response.choices[0].message.content)

complaint


In [12]:
import json

messages = [
    {"role": "system", "content": 'Extract the order. Return ONLY JSON with keys: name, item, qty, date, eggless. Use a number for qty and true/false for eggless.'},
    {"role": "user", "content": "My name is Purnima. I want 2 plum cakes for Friday, eggless."}
]

response = client.chat.completions.create(
  model=MODEL_NAME,
  messages=messages,
  temperature=0.2
)
reply = response.choices[0].message.content
try:
   order = json.loads(reply)
   print(order)
   print("Item:", order["item"])
   print("Quantity:", order["qty"])
except json.JSONDecodeError:
    print("The response was not valid JSON:")
    print(reply)

{'name': 'Purnima', 'item': 'plum cake', 'qty': 2, 'date': 'Friday', 'eggless': True}
Item: plum cake
Quantity: 2
